In [ ]:
!pip install pandas jupyter

In [ ]:
import pandas as pd
import json
import csv

In [ ]:
csv_file_path = 'main_df.csv'
df_csv = pd.read_csv(csv_file_path)
df_csv.head()
df_csv.info()

In [ ]:
df_csv = df_csv.drop(columns=["DBA Name", "City", "Zip Code", "Address", "CountyFIPS",])
df_csv.head()

In [ ]:
df_csv = df_csv.rename(columns={
    "License Number": "id",
    "Normalized DBA Name": "names",
    "Is Chain Store": "chain",
    "BoroName": "borough",
    "NTAName": "neighborhood",
    "Name Words": "words"
})
df_csv.head()

In [ ]:
json_file_path = 'store_sign_flat.json'
with open(json_file_path, 'r') as file:
    data_json = json.load(file)
df_json = pd.DataFrame(data_json)
df_json.info()

In [ ]:
df_json.head()

In [ ]:
merged_df = pd.merge(df_csv, df_json, on='id', how='inner')
merged_df.head()

In [ ]:
merged_df.info()

In [ ]:
import ast

# Convert string representation of lists to actual lists
merged_df['words'] = merged_df['words'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') and x.endswith(']') else x
)

# Verify the changes
print(merged_df['words'].head())

In [ ]:
# Load the nbh_location.csv file
nbh_location_file_path = 'nbh_location.csv'
nbh_location_df = pd.read_csv(nbh_location_file_path)

# Perform the join operation
merged_df = pd.merge(merged_df, nbh_location_df[['License Number', 'CDTA2020']], 
                     left_on='id', right_on='License Number', how='inner')

# Drop the redundant 'License Number' column
merged_df = merged_df.drop(columns=['License Number'])

# Verify the changes
merged_df

In [ ]:
null_code_df = merged_df[merged_df['CDTA2020'].isnull()]
null_code_df


In [ ]:
merged_df = merged_df.drop(columns=["NTA2020"])
merged_df = merged_df.rename(columns={"CDTA2020": "neighborhood_code"})

# Verify the changes
merged_df.head()

In [ ]:
merged_df.to_json('merged_df.json', orient="records", indent=2)